# 03 - Foundation Models: SAM 2 & CLIP

## From Object Detection to Pixel-Perfect Understanding

In the previous notebooks, we went from raw pixels (OpenCV) to bounding boxes (YOLO). Now we take the final leap into **Foundation Models** — massive, general-purpose AI systems that can do things they were never explicitly taught.

We will explore three ideas:

| Task | Model | What It Does |
|---|---|---|
| **Automatic Mask Generation** | Meta SAM 2 | Finds *every* object and draws a pixel-perfect mask around it |
| **Zero-Shot Classification** | OpenAI CLIP | Classifies images using plain English — no training needed |
| **The Mega-Pipeline** | YOLO + SAM 2 | Uses YOLO for speed and SAM 2 for precision — the industry standard |

> **Hardware Note:** SAM 2 and CLIP run on CPU but are noticeably faster on a GPU. On a laptop CPU, expect ~3-10 seconds per image for SAM 2's automatic mask generation.

## Setup: Download the SAM 2 Model Checkpoint

SAM 2 needs a pre-trained checkpoint and a model config. We'll use the **SAM 2.1 Hiera Small** variant (~150 MB) — fast enough for a laptop workshop. The file will be saved into the `models/` folder.

In [ ]:
import os, urllib.request

SAM2_CHECKPOINT = "models/sam2.1_hiera_small.pt"
SAM2_URL = "https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_small.pt"
SAM2_CONFIG = "configs/sam2.1/sam2.1_hiera_s.yaml"

os.makedirs("models", exist_ok=True)

if not os.path.exists(SAM2_CHECKPOINT):
    print("Downloading SAM 2.1 Hiera Small checkpoint (~150 MB)...")
    urllib.request.urlretrieve(SAM2_URL, SAM2_CHECKPOINT)
    print(f"Done! Saved to {SAM2_CHECKPOINT}")
else:
    print(f"Checkpoint already exists at {SAM2_CHECKPOINT}")

## Helper Functions for Visualisation

These helpers overlay coloured masks and bounding boxes on images using Matplotlib. We'll reuse them throughout the notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
from PIL import Image

def show_mask(mask, ax, random_color=True):
    """Overlay a single binary mask on a Matplotlib axis."""
    if random_color:
        color = np.concatenate([np.random.random(3), np.array([0.5])], axis=0)
    else:
        color = np.array([30/255, 144/255, 255/255, 0.5])
    h, w = mask.shape[-2:]
    mask_image = mask.reshape(h, w, 1) * color.reshape(1, 1, -1)
    ax.imshow(mask_image)

def show_box(box, ax, label=None, color='lime'):
    """Draw a bounding box on a Matplotlib axis."""
    x0, y0, x1, y1 = box
    rect = plt.Rectangle((x0, y0), x1 - x0, y1 - y0,
                          edgecolor=color, facecolor='none', linewidth=2)
    ax.add_patch(rect)
    if label:
        ax.text(x0, y0 - 6, label, color=color, fontsize=9,
                fontweight='bold', backgroundcolor='black')

def show_all_masks(image, masks, title="SAM Automatic Masks"):
    """Overlay all masks from SAM's automatic generator on the image."""
    plt.figure(figsize=(12, 8))
    plt.imshow(image)
    # Sort by area so large masks are painted first (small ones on top)
    sorted_masks = sorted(masks, key=lambda x: x['area'], reverse=True)
    for mask_data in sorted_masks:
        show_mask(mask_data['segmentation'], plt.gca())
    plt.title(title)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

print("Helpers loaded!")

## Load a Sample Image

We'll use our sample image from the `images/` folder. SAM expects a standard NumPy RGB array.

In [ ]:
# Load image — OpenCV reads as BGR, so we convert to RGB for SAM & Matplotlib
image_bgr = cv2.imread("samples/images/kookaburra.jpeg")
image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

print(f"Image shape: {image_rgb.shape}  (Height x Width x Channels)")

plt.figure(figsize=(10, 7))
plt.imshow(image_rgb)
plt.title("Sample Image — Kookaburra")
plt.axis('off')
plt.show()

---

## Task 1: Automatic Mask Generation with SAM 2

SAM 2's `SAM2AutomaticMaskGenerator` places a grid of point prompts across the entire image and asks: *"Is there an object here?"* for every single point. The result is a list of **every distinct region** SAM 2 can find.

### How It Works
1. A grid of points is scattered over the image (controlled by `points_per_side`).
2. For each point, SAM 2 predicts one or more masks.
3. Overlapping / low-quality masks are filtered by IoU and stability thresholds.
4. What's left is a clean set of masks for every "thing" in the scene.

In [ ]:
import torch
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator

# 1. Choose the best available device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# 2. Load the SAM 2 model from checkpoint + config
sam2_model = build_sam2(SAM2_CONFIG, SAM2_CHECKPOINT, device=device)
print("SAM 2.1 Hiera Small model loaded!")

# 3. Create the Automatic Mask Generator
#    - points_per_side: density of the prompt grid (32 = 32x32 = 1024 points)
#    - pred_iou_thresh: discard masks below this predicted quality
#    - stability_score_thresh: discard masks that change too much with small input tweaks
mask_generator = SAM2AutomaticMaskGenerator(
    model=sam2_model,
    points_per_side=32,
    pred_iou_thresh=0.86,
    stability_score_thresh=0.92,
)
print("Mask generator ready!")

In [ ]:
# 4. Run the generator on our image
# This is the "expensive" step — it scatters 1024 points and runs SAM 2 for each one
print("Generating masks (this may take 3-15 seconds on CPU)...")

masks = mask_generator.generate(image_rgb)

print(f"\nSAM 2 found {len(masks)} distinct regions!")
print(f"\nEach mask is a dictionary with keys: {list(masks[0].keys())}")
print(f"  - 'segmentation': a {masks[0]['segmentation'].shape} boolean array (the pixel mask)")
print(f"  - 'area': {masks[0]['area']} pixels")
print(f"  - 'predicted_iou': {masks[0]['predicted_iou']:.3f}")

In [ ]:
# 5. Visualise ALL the masks painted onto the image
show_all_masks(image_rgb, masks, title=f"SAM Automatic Mask Generation — {len(masks)} masks")

In [ ]:
# 6. Zoom in: show the top-5 largest masks individually
sorted_masks = sorted(masks, key=lambda x: x['area'], reverse=True)

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for i, ax in enumerate(axes):
    ax.imshow(image_rgb)
    show_mask(sorted_masks[i]['segmentation'], ax, random_color=True)
    ax.set_title(f"Mask #{i+1}  —  {sorted_masks[i]['area']:,} px")
    ax.axis('off')

plt.suptitle("Top 5 Largest Masks (Individual)", fontsize=14) 
plt.tight_layout()
plt.show()

---

## Task 2: Zero-Shot Classification with CLIP

**CLIP** (Contrastive Language–Image Pre-training) was trained on 400 million image-text pairs scraped from the internet. It learned to map images and text into the **same mathematical space**, meaning we can compare any image to any text description and ask: *"How similar are these?"*

This is called **"Open Vocabulary"** classification — you don't train it, you just describe what you're looking for in plain English.

### How It Works
1. Encode the image into a vector (list of numbers).
2. Encode each candidate text label into a vector.
3. Compute the cosine similarity between the image vector and every text vector.
4. The highest similarity = the best match. No training data required!

In [ ]:
from transformers import CLIPProcessor, CLIPModel

# 1. Load CLIP (downloads ~600 MB on first run, then cached)
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
print("CLIP model loaded!")

In [ ]:
# 2. Define candidate labels — mix of correct answers and distractors
candidate_labels = [
    "a photo of a kookaburra bird",		# try commenting this out to see how it affects results!
    "a photo of a bald eagle",
    "a photo of a cat sitting on a fence",
    "a photo of a 1960s blue Mustang",
    "a photo of a bird on a tree branch",
    "a photo of a dog playing fetch",
    "a photo of a tropical sunset"
]

# 3. Run CLIP — encode the image and all text labels at once
pil_image = Image.fromarray(image_rgb)
inputs = clip_processor(text=candidate_labels, images=pil_image, return_tensors="pt", padding=True)

with torch.no_grad():
    outputs = clip_model(**inputs)

# 4. Get similarity scores and softmax to probabilities
logits = outputs.logits_per_image[0]            # shape: (num_labels,)
probs = logits.softmax(dim=0).cpu().numpy()     # convert to probabilities

# 5. Display results
print("CLIP Zero-Shot Classification Results:\n")
for label, prob in sorted(zip(candidate_labels, probs), key=lambda x: x[1], reverse=True):
    bar = "█" * int(prob * 50)
    print(f"  {prob*100:5.1f}%  {bar}  {label}")

In [ ]:
# 6. Visualise as a horizontal bar chart
fig, (ax_img, ax_bar) = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={'width_ratios': [1, 1.5]})

ax_img.imshow(image_rgb)
ax_img.set_title("Input Image")
ax_img.axis('off')

# Sort for the chart
sorted_pairs = sorted(zip(candidate_labels, probs), key=lambda x: x[1])
labels_sorted = [p[0] for p in sorted_pairs]
probs_sorted = [p[1] for p in sorted_pairs]
colors = ['#2ecc71' if p == max(probs_sorted) else '#3498db' for p in probs_sorted]

ax_bar.barh(labels_sorted, probs_sorted, color=colors)
ax_bar.set_xlabel("Probability")
ax_bar.set_title("CLIP Zero-Shot Classification")
ax_bar.set_xlim(0, 1)

for i, (p, label) in enumerate(zip(probs_sorted, labels_sorted)):
    ax_bar.text(p + 0.01, i, f"{p*100:.1f}%", va='center', fontsize=9)

plt.tight_layout()
plt.show()

### Experiment: Open-Vocabulary Power

Try changing the labels above to anything you can imagine — fine-grained descriptions, abstract concepts, or artistic styles. CLIP has never been trained on your specific labels; it *understands* them from its massive pre-training. That's the magic of foundation models.

---

## Task 3: The Mega-Pipeline — YOLO + SAM 2

In production, nobody uses SAM’s automatic mode to segment specific objects. Instead, the industry-standard workflow is:

1. **YOLO** detects objects in milliseconds and gives bounding boxes + class labels.
2. **SAM 2** takes each bounding box as a *prompt* and refines it into a pixel-perfect mask.

This gives you the **speed of YOLO** combined with the **precision of SAM 2**. It's the go-to approach for:
- Creating high-quality training datasets
- Precise background removal
- Medical / satellite image annotation
- Robotics and autonomous systems

We'll build this pipeline as a reusable function, then run it on our kookaburra image and finally on a video.

In [ ]:
from ultralytics import YOLO

# 1. Load YOLOv8 (we already have the weights from Notebook 02)
yolo_model = YOLO('models/yolov8n.pt')

# 2. Set up SAM 2 in "Image Predictor" mode (accepts box prompts instead of auto-generating)
sam2_predictor = SAM2ImagePredictor(sam2_model)
print("YOLO + SAM 2 Predictor ready!")

In [ ]:
def yolo_sam_pipeline(image_rgb, yolo_model, sam2_predictor, conf=0.5):
    """
    Run YOLO detection then refine each box with SAM 2.
    
    Returns:
        results_list: list of dicts with keys 'box', 'label', 'confidence', 'mask'
        yolo_results: raw YOLO results (for the annotated frame)
    """
    # Step 1 — YOLO detects objects (fast: ~5-20 ms)
    yolo_results = yolo_model(image_rgb, conf=conf, verbose=False)
    
    results_list = []
    
    if yolo_results[0].boxes is not None and len(yolo_results[0].boxes) > 0:
        boxes = yolo_results[0].boxes.xyxy.cpu().numpy()      # [x1, y1, x2, y2]
        classes = yolo_results[0].boxes.cls.cpu().numpy()
        confs = yolo_results[0].boxes.conf.cpu().numpy()
        names = yolo_results[0].names  # class ID → name mapping
        
        # Step 2 — Feed the image into SAM 2's encoder once
        with torch.inference_mode():
            sam2_predictor.set_image(image_rgb)
            
            for box, cls_id, confidence in zip(boxes, classes, confs):
                # Step 3 — SAM 2 refines each YOLO box into a pixel-perfect mask
                sam_masks, sam_scores, _ = sam2_predictor.predict(
                    box=box,
                    multimask_output=False  # We only want the single best mask per box
                )
                
                # SAM 2 may return float masks — threshold to boolean
                mask = sam_masks[0] > 0.5
                
                results_list.append({
                    'box': box,
                    'label': names[int(cls_id)],
                    'confidence': float(confidence),
                    'mask': mask,  # boolean array (H, W)
                    'sam_score': float(sam_scores[0]),
                })
    
    return results_list, yolo_results

print("Pipeline function defined!")

In [ ]:
# 3. Run the Mega-Pipeline on our sample image
pipeline_results, yolo_raw = yolo_sam_pipeline(image_rgb, yolo_model, sam2_predictor, conf=0.4)

print(f"YOLO detected {len(pipeline_results)} object(s):\n")
for i, r in enumerate(pipeline_results):
    print(f"  [{i+1}] {r['label']} — YOLO conf: {r['confidence']:.2f}, SAM quality: {r['sam_score']:.3f}")

In [ ]:
# 4. Visualise side-by-side: YOLO boxes vs SAM masks
fig, (ax_yolo, ax_sam) = plt.subplots(1, 2, figsize=(18, 8))

# Left panel — YOLO bounding boxes only
ax_yolo.imshow(image_rgb)
for r in pipeline_results:
    show_box(r['box'], ax_yolo, label=f"{r['label']} {r['confidence']:.0%}")
ax_yolo.set_title("YOLO Detection (Bounding Boxes)", fontsize=13)
ax_yolo.axis('off')

# Right panel — SAM masks with labels
ax_sam.imshow(image_rgb)
for r in pipeline_results:
    show_mask(r['mask'], ax_sam)
    show_box(r['box'], ax_sam, label=f"{r['label']} {r['confidence']:.0%}", color='white')
ax_sam.set_title("YOLO + SAM 2 (Pixel-Perfect Masks)", fontsize=13)
ax_sam.axis('off')

plt.suptitle("The Mega-Pipeline: Boxes → Masks", fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

---

## Bonus: Run the Mega-Pipeline on Video

We can apply the same YOLO → SAM 2 pipeline to a video. Pick an `.mp4` file from the `samples/videos/` directory (or drop in your own) and set the path below — or set `USE_WEBCAM = True` to use your camera instead.

> **Performance Note:** SAM 2 is significantly faster than the original SAM, but the image encoder still takes ~0.5-2 seconds per frame on CPU. For the workshop, we process every $N^{\text{th}}$ frame and write the results to an output file.

In [ ]:
# ============================================================
# CONFIG — Change these to switch between webcam and video file
# ============================================================
USE_WEBCAM = False                                   # Set True for webcam
VIDEO_PATH = "samples/videos/car.mp4"         		 # Path to your video file
PROCESS_EVERY_N = 1                                  # Process every Nth frame (speed vs quality)
MAX_FRAMES = 1000                                    # Cap total frames to process (0 = no limit)
# ============================================================

import os
os.makedirs("results", exist_ok=True)

# Build output filename: output_<input_filename>.mp4
video_basename = os.path.splitext(os.path.basename(VIDEO_PATH))[0]
OUTPUT_PATH = f"results/output_{video_basename}.mp4"

# Select input source
if USE_WEBCAM:
    cap = cv2.VideoCapture(0)
    OUTPUT_PATH = "results/output_webcam.mp4"
    print("Using WEBCAM as input source.")
else:
    if not os.path.exists(VIDEO_PATH):
        print(f"⚠️  Video not found at '{VIDEO_PATH}'.")
        print("   Drop a .mp4 file into samples/videos/ and update VIDEO_PATH above.")
        cap = None
    else:
        cap = cv2.VideoCapture(VIDEO_PATH)
        print(f"Using video file: {VIDEO_PATH}")

print(f"Output will be saved to: {OUTPUT_PATH}")

if cap is not None and cap.isOpened():
    # Get video properties
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    print(f"Video: {w}x{h} @ {fps:.0f} FPS, {total} total frames")
    print(f"Processing every {PROCESS_EVERY_N}th frame (≈{total // PROCESS_EVERY_N} frames to process)")
    
    # Setup video writer
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(OUTPUT_PATH, fourcc, fps / PROCESS_EVERY_N, (w, h))
    
    frame_idx = 0
    processed = 0
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        frame_idx += 1
        
        # Skip frames we don't want to process
        if frame_idx % PROCESS_EVERY_N != 0:
            continue
        
        # Convert BGR → RGB for SAM
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        
        # Run the YOLO + SAM 2 mega-pipeline
        results_list, _ = yolo_sam_pipeline(frame_rgb, yolo_model, sam2_predictor, conf=0.5)
        
        # Draw masks and labels on the frame
        overlay = frame.copy()
        for r in results_list:
            # Create a coloured mask overlay
            color = np.random.randint(50, 255, 3).tolist()
            mask_bool = r['mask']
            overlay[mask_bool] = (
                np.array(overlay[mask_bool], dtype=np.float32) * 0.5 +
                np.array(color, dtype=np.float32) * 0.5
            ).astype(np.uint8)
            
            # Draw the box and label
            x1, y1, x2, y2 = r['box'].astype(int)
            cv2.rectangle(overlay, (x1, y1), (x2, y2), color, 2)
            cv2.putText(overlay, f"{r['label']} {r['confidence']:.0%}",
                        (x1, y1 - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
        
        out.write(overlay)
        processed += 1
        
        if processed % 5 == 0:
            print(f"  Processed {processed} frames...")
        
        if MAX_FRAMES > 0 and processed >= MAX_FRAMES:
            print(f"  Reached MAX_FRAMES limit ({MAX_FRAMES}).")
            break
    
    cap.release()
    out.release()
    print(f"\n✅ Done! Output saved to {OUTPUT_PATH} ({processed} frames)")
else:
    if cap is not None:
        print("Could not open video source.")

In [ ]:
# Preview the output video — show a few sample frames
if os.path.exists(OUTPUT_PATH):
    preview_cap = cv2.VideoCapture(OUTPUT_PATH)
    sample_frames = []
    
    total_out = int(preview_cap.get(cv2.CAP_PROP_FRAME_COUNT))
    step = max(1, total_out // 4)  # pick ~4 evenly spaced frames
    
    for i in range(0, total_out, step):
        preview_cap.set(cv2.CAP_PROP_POS_FRAMES, i)
        ret, frame = preview_cap.read()
        if ret:
            sample_frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    preview_cap.release()
    
    if sample_frames:
        fig, axes = plt.subplots(1, len(sample_frames), figsize=(5 * len(sample_frames), 5))
        if len(sample_frames) == 1:
            axes = [axes]
        for ax, f in zip(axes, sample_frames):
            ax.imshow(f)
            ax.axis('off')
        plt.suptitle("Sample Frames from Mega-Pipeline Output", fontsize=14)
        plt.tight_layout()
        plt.show()
else:
    print("No output video found — run the cell above first.")

---

## Recap

| What We Did | Model | Key Takeaway |
|---|---|---|
| **Automatic Mask Generation** | SAM 2 | Finds every object without any class labels — pure segmentation |
| **Zero-Shot Classification** | CLIP | Classifies images using plain text prompts — no training needed |
| **Mega-Pipeline** | YOLO + SAM 2 | YOLO for speed, SAM 2 for precision — the industry standard |

### What's Next?
- **SAM 2 Video Predictor**: Use `SAM2VideoPredictor` to track and segment objects across an entire video with a single prompt — no per-frame processing needed.
- **Grounding DINO + SAM 2**: Replace YOLO with a text-prompted detector — "segment the coffee mug" without any training.
- **Fine-tuning CLIP**: Adapt CLIP to your own domain for even more accurate zero-shot classification.

Congratulations — you've gone from raw pixels all the way through to foundation model pipelines!